In [32]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

from sklearn.model_selection import TimeSeriesSplit, cross_val_score, GridSearchCV

In [33]:
data_proc = pd.read_csv('../data/processed/data_processed.csv')

## Podział danych na testowe i treningowe

Podział na 80% danych treningowych i 20% testowych, jednocześnie chcemy, aby dane testowe były najnowszymi danymi (patrząc po zmiennej 'created_at').

In [34]:
def prepare_time_based_split(df, target_col='price_per_m', date_col='created_at', train_size=0.8):

    df = df.sort_values(date_col, ascending=True)
    split_point = int(len(df) * train_size)
    train_df = df.iloc[:split_point]
    test_df = df.iloc[split_point:]

    features_to_drop = ['created_at', 'created_at_first', target_col] # kolumny z datami nie będą juz uzywane
    
    X_train = train_df.drop(columns=features_to_drop)
    y_train = train_df[target_col]
    
    X_test = test_df.drop(columns=features_to_drop)
    y_test = test_df[target_col]
    
    return X_train, X_test, y_train, y_test

In [35]:
X_train_full, X_test_full, y_train, y_test = prepare_time_based_split(data_proc)

In [36]:
# zmienne do baseline
data_base = ['district_lon', 'district_lat', 'no_rooms', 'm', 'map_lon', 'map_lat']

X_train_base = X_train_full[data_base]
X_test_base = X_test_full[data_base]

## Dane lingwistyczne - Tfidf

In [37]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=5, max_df=0.9)

X_train_tfidf = tfidf.fit_transform(X_train_full['clean_description']) # trenujemy na zbiorze treningowym
X_test_tfidf = tfidf.transform(X_test_full['clean_description']) # transformujemy zbiór testowy

# redukcja wymiarów
n_components = 60
svd = TruncatedSVD(n_components=n_components, random_state=443712)
X_train_lsa = svd.fit_transform(X_train_tfidf)

X_test_lsa = svd.transform(X_test_tfidf) # zbiór testowy transformujemy

In [38]:
lsa_columns = [f'tfidf_lsa_{i}' for i in range(n_components)]

X_train_lsa_df = pd.DataFrame(X_train_lsa, columns=lsa_columns, index=X_train_full.index)
X_test_lsa_df = pd.DataFrame(X_test_lsa, columns=lsa_columns, index=X_test_full.index)

X_train_full = pd.concat([X_train_full, X_train_lsa_df], axis=1)
X_test_full = pd.concat([X_test_full, X_test_lsa_df], axis=1)

In [39]:
X_train_full['clean_title'] = X_train_full['clean_title'].fillna("")
X_test_full['clean_title'] = X_test_full['clean_title'].fillna("")

X_train_full['clean_title'] = X_train_full['clean_title'].astype(str)
X_test_full['clean_title'] = X_test_full['clean_title'].astype(str)

In [40]:
tfidf2 = TfidfVectorizer(max_features=500, ngram_range=(1, 2), min_df=5, max_df=0.9)

X_train_tfidf2 = tfidf2.fit_transform(X_train_full['clean_title'])
X_test_tfidf2 = tfidf2.transform(X_test_full['clean_title'])

# redukcja wymiarów
n_components = 30
svd2 = TruncatedSVD(n_components=n_components, random_state=443712)
X_train_lsa2 = svd2.fit_transform(X_train_tfidf2)

X_test_lsa2 = svd2.transform(X_test_tfidf2) # zbiór testowy transformujemy

lsa_columns_title = [f'tfidf_title_lsa_{i}' for i in range(n_components)]

X_train_lsa_df2 = pd.DataFrame(X_train_lsa2, columns=lsa_columns_title, index=X_train_full.index)
X_test_lsa_df2 = pd.DataFrame(X_test_lsa2, columns=lsa_columns_title, index=X_test_full.index)

X_train_full = pd.concat([X_train_full, X_train_lsa_df2], axis=1)
X_test_full = pd.concat([X_test_full, X_test_lsa_df2], axis=1)

In [41]:
# usuwamy zmienne ['clean_description', 'extras_', 'clean_title'] ze zbioru treningowego i testowego
col = ['clean_description', 'extras_', 'clean_title']
X_train_full = X_train_full.drop(columns = col, errors = 'ignore')
X_test_full = X_test_full.drop(columns = col, errors = 'ignore')

## Modelowanie - grid_search

Będziemy testować trzy modele Random Forest, XGBoost oraz LightGBM.

In [42]:
def evaluate_model(y_true, y_pred, model_name, dataset_name):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    print(f"Wyniki na zbiorze testowym {model_name} ({dataset_name})")
    print(f"R^2:  {r2:.4f}")
    print(f"MAE:  {mae:.2f} PLN")
    print(f"RMSE: {rmse:.2f} PLN")
    print(f"MAPE: {mape:.2%}")

    return {"R2": r2, "MAE": mae, "RMSE": rmse, "MAPE": mape}

In [43]:
tscv = TimeSeriesSplit(n_splits=3)

### Baseline

In [44]:
models_config = [{"name": "Random Forest", "estimator": RandomForestRegressor(random_state=441273, n_jobs=-1),
     "param_grid": {'n_estimators': [100, 300, 500], 'max_depth': [10, 20, None], 'min_samples_split': [2, 5, 10]}},
    {"name": "XGBoost", "estimator": xgb.XGBRegressor(random_state=441273, n_jobs=-1),
     "param_grid": {'n_estimators': [100, 300, 500], 'max_depth': [5, 10, 15], 'learning_rate': [0.01, 0.05, 0.1, 0.5]}},
    {"name": "LightGBM", "estimator": lgb.LGBMRegressor(random_state=441273, n_jobs=-1),
     "param_grid": {'n_estimators': [100, 300, 500], 'max_depth': [5, 10, -1], 'learning_rate': [0.01, 0.05, 0.1]}}]

results_list = []

for model_info in models_config:
    name = model_info["name"]

    grid_search = GridSearchCV(estimator=model_info["estimator"], param_grid=model_info["param_grid"], cv=tscv,
                               scoring='neg_mean_absolute_error', n_jobs=-1)
    grid_search.fit(X_train_base, y_train)

    print(f"\nNajlepsze parametry dla {name}:")
    print(grid_search.best_params_)

    best_cv_mae = -grid_search.best_score_
    y_pred = grid_search.predict(X_test_base)
    metrics = evaluate_model(y_test, y_pred, name, "Full")

    results_list.append({"Model": name, "Test_MAE": metrics["MAE"], "Test_MAPE_%": metrics["MAPE"] * 100, "Test_R2": metrics["R2"],
        "CV_MAE": best_cv_mae, "Params": str(grid_search.best_params_)})

df_results = pd.DataFrame(results_list).sort_values("Test_MAE")
pd.set_option('display.max_colwidth', None)
display(df_results)


Najlepsze parametry dla Random Forest:
{'max_depth': None, 'min_samples_split': 10, 'n_estimators': 500}
Wyniki na zbiorze testowym Random Forest (Full)
R^2:  0.4064
MAE:  478.44 PLN
RMSE: 640.77 PLN
MAPE: 6.03%


/Users/gabrielajeznach/Envs/moj_python_12/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



Najlepsze parametry dla XGBoost:
{'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 300}
Wyniki na zbiorze testowym XGBoost (Full)
R^2:  0.3643
MAE:  506.09 PLN
RMSE: 663.11 PLN
MAPE: 6.36%
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000271 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 800
[LightGBM] [Info] Number of data points in the train set: 6075, number of used features: 6
[LightGBM] [Info] Start training from score 7184.025909
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001026 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 794
[LightGBM] [Info] Number of data points in the train set: 4050, number of used features: 6
[LightGBM] [Info] Start training from score 6955.979052
[LightGBM] [Info] Auto-choosing col-wise multi-threading

/Users/gabrielajeznach/Envs/moj_python_12/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

,Model,Test_MAE,Test_MAPE_%,Test_R2,CV_MAE,Params
0,Random Forest,478.439178,6.028644,0.406360,592.564035,"{'max_depth': None, 'min_samples_split': 10, 'n_estimators': 500}"
1,XGBoost,506.088287,6.359014,0.364252,603.256009,"{'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 300}"
2,LightGBM,513.478358,6.448828,0.366618,607.312546,"{'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100}"


In [45]:
df_results.to_csv("base_results_tfidf.csv")

### Pełny zbiór

In [ ]:
models_config = [{"name": "Random Forest", "estimator": RandomForestRegressor(random_state=441273, n_jobs=-1),
     "param_grid": {'n_estimators': [100, 300, 500], 'max_depth': [10, 20, None], 'min_samples_split': [2, 5, 10]}},
    {"name": "XGBoost", "estimator": xgb.XGBRegressor(random_state=441273, n_jobs=-1),
     "param_grid": {'n_estimators': [100, 300, 500], 'max_depth': [5, 10, 15], 'learning_rate': [0.01, 0.05, 0.1, 0.5]}},
    {"name": "LightGBM", "estimator": lgb.LGBMRegressor(random_state=441273, n_jobs=-1),
     "param_grid": {'n_estimators': [100, 300, 500], 'max_depth': [5, 10, -1], 'learning_rate': [0.01, 0.05, 0.1]}}]

results_list = []

for model_info in models_config:
    name = model_info["name"]

    grid_search = GridSearchCV(estimator=model_info["estimator"], param_grid=model_info["param_grid"], cv=tscv,
                               scoring='neg_mean_absolute_error', n_jobs=-1)
    grid_search.fit(X_train_full, y_train)

    best_cv_mae = -grid_search.best_score_
    y_pred = grid_search.predict(X_test_full)
    metrics = evaluate_model(y_test, y_pred, name, "Full")
    results_list.append({"Model": name, "Test_MAE": metrics["MAE"], "Test_MAPE_%": metrics["MAPE"] * 100, "Test_R2": metrics["R2"],
        "CV_MAE": best_cv_mae, "Params": str(grid_search.best_params_)})

df_results = pd.DataFrame(results_list).sort_values("Test_MAE")
pd.set_option('display.max_colwidth', None)
display(df_results)

In [ ]:
df_results.to_csv("full_results_tfidf.csv")
# Ostatecznie wybieramy model XGBoost z parametrami {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 500}.